In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

In [2]:
# --- Load data ---
expr = pd.read_csv('../data/raw/HiSeqV2', sep='\t', index_col=0)
mut = pd.read_csv('../data/raw/BRCA_mc3_gene_level.txt', sep='\t', index_col=0)
clin = pd.read_csv('../data/raw/TCGA.BRCA.sampleMap_BRCA_clinicalMatrix', sep='\t', index_col=0)

In [3]:
# --- Find common patients with PAM50 label ---
expr_samples = set(expr.columns)
mut_samples = set(mut.columns)
labeled_patients = set(clin[clin['PAM50Call_RNAseq'].notna()].index)
common = expr_samples & mut_samples & labeled_patients
common_patients = sorted(common)

In [4]:
# --- Subset and transpose ---
expr_subset = expr[common_patients].T
mut_subset = mut[common_patients].T
labels = clin.loc[common_patients, 'PAM50Call_RNAseq']

print("Expression subset:", expr_subset.shape)
print("Mutation subset:", mut_subset.shape)
print("Labels:", labels.shape)

Expression subset: (593, 20530)
Mutation subset: (593, 40543)
Labels: (593,)


In [5]:
# --- Now feature filtering ---
gene_variances = expr_subset.var(axis=0)
top_n = 2000
top_genes = gene_variances.sort_values(ascending=False).head(top_n).index
expr_filtered = expr_subset[top_genes]
print("Expression filtered:", expr_filtered.shape)

# Stricter mutation filter: keep genes mutated in at least 1% of patients
min_patients = int(0.01 * len(mut_subset))  # ~6 patients
mut_filtered = mut_subset.loc[:, (mut_subset.sum(axis=0) >= min_patients)]
print("Mutation filtered (stricter):", mut_filtered.shape)
print("Min patients threshold:", min_patients)

le = LabelEncoder()
y = le.fit_transform(labels)
print("Classes:", le.classes_)
print("Encoded label counts:", np.bincount(y))

Expression filtered: (593, 2000)
Mutation filtered (stricter): (593, 2714)
Min patients threshold: 5
Classes: ['Basal' 'Her2' 'LumA' 'LumB' 'Normal']
Encoded label counts: [106  42 301 125  19]


In [6]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

In [7]:
# Scale expression (z-score per gene)
scaler = StandardScaler()
expr_scaled = scaler.fit_transform(expr_filtered)
print("Expression scaled:", expr_scaled.shape)

Expression scaled: (593, 2000)


In [8]:
# Train/val/test split (70/15/15, stratified)
X_expr_train, X_expr_temp, X_mut_train, X_mut_temp, y_train, y_temp = train_test_split(
    expr_scaled, mut_filtered.values, y, test_size=0.3, stratify=y, random_state=42
)
X_expr_val, X_expr_test, X_mut_val, X_mut_test, y_val, y_test = train_test_split(
    X_expr_temp, X_mut_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42
)

print("Train:", X_expr_train.shape, X_mut_train.shape, y_train.shape)
print("Val:", X_expr_val.shape, X_mut_val.shape, y_val.shape)
print("Test:", X_expr_test.shape, X_mut_test.shape, y_test.shape)

Train: (415, 2000) (415, 2714) (415,)
Val: (89, 2000) (89, 2714) (89,)
Test: (89, 2000) (89, 2714) (89,)


In [9]:
# Check class distribution preserved across splits
print("Train class counts:", np.bincount(y_train))
print("Val class counts:", np.bincount(y_val))
print("Test class counts:", np.bincount(y_test))

Train class counts: [ 74  29 211  88  13]
Val class counts: [16  6 45 19  3]
Test class counts: [16  7 45 18  3]
